In [12]:
import cvxpy as cp
import itertools
import numpy as np
import pandas as pd
import time
import tqdm
import warnings

warnings.filterwarnings("ignore")

In [13]:
def normalize_priors(priors):
    if np.sum(priors) == 0:
        return np.ones_like(priors, dtype=float) / len(priors)
    return priors / np.sum(priors)

def best_response(x, thresholds, priors, c):
    posteriors = normalize_priors(priors)
    search_space = [x] + thresholds[thresholds > x].tolist()
    utilities = []
    for i, x_p in enumerate(search_space):
        utility = np.dot(posteriors, x_p >= thresholds)
        cost = c * abs(x-x_p)
        utilities.append(utility - cost + ((len(search_space)-i)*1e-6))
    return search_space[np.argmax(utilities)]

def best_response_vectorized(X, thresholds, priors, c):
    posteriors = normalize_priors(priors).flatten()
    utilities_expected = np.array([
        np.dot(posteriors, thresholds[j] >= thresholds)
        for j in range(len(thresholds))
    ])

    pass_matrix = X[:, None] >= thresholds[None, :]
    utility_stay = pass_matrix @ posteriors

    X_col = X[:, None]
    thresholds_row = thresholds[None, :]

    feasible = thresholds_row > X_col
    utility_jump = utilities_expected[None, :] - c * np.abs(thresholds_row - X_col)
    utility_jump[~feasible] = -np.inf

    utilities = np.concatenate([utility_stay[:, None], utility_jump], axis=1)
    best_idx = np.argmax(utilities + np.array([(utilities.shape[1] - i) * 1e-6 for i in range(utilities.shape[1])]), axis=1)
    X_p = np.where(best_idx == 0, X, thresholds[best_idx - 1])
    return X_p

def accuracy_loss_vectorized(X, X_p, thresholds, priors, threshold_true, return_breakdowns=False):
    Y_true = (X >= threshold_true).astype(float)
    Y_p = (X_p[:, None] >= thresholds[None, :]).astype(float)
    losses = np.abs(Y_true[:, None] - Y_p).mean(axis=0)
    posteriors = normalize_priors(priors)
    if return_breakdowns:
        return np.dot(losses, posteriors), losses * priors
    else:
        return np.dot(losses, posteriors)

def evaluate_conditional(X, partition, thresholds, priors, threshold_true, c, return_breakdowns=False):
    thresholds_p = thresholds[partition]
    priors_p     = priors[partition]
    X_p = best_response_vectorized(X, thresholds_p, priors_p, c)
    return accuracy_loss_vectorized(X, X_p, thresholds_p, priors_p, threshold_true, return_breakdowns)

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c, return_breakdowns=False):
    acc_loss = 0.0
    acc_loss_list = []
    for partition in partitions:
        if return_breakdowns:
            acc_loss_p, acc_loss_p_list = evaluate_conditional(X, partition, thresholds, priors, threshold_true, c, return_breakdowns)
            acc_loss_list.append(acc_loss_p_list.tolist())
        else:
            acc_loss_p = evaluate_conditional(X, partition, thresholds, priors, threshold_true, c, return_breakdowns)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    if return_breakdowns:
        return acc_loss, acc_loss_list
    else:
        return acc_loss

In [14]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return
    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
    t0 = time.perf_counter()
    indices = list(range(len(thresholds)))
    best_partition, best_loss = None, np.inf
    for partition in set_partitions(indices):
        acc_loss = evaluate_system(X, partition, thresholds, priors, threshold_true, c)
        if acc_loss <= best_loss:
            if best_loss - acc_loss < 1e-9:
                if len(partition) < len(best_partition):
                    best_loss = acc_loss
                    best_partition = partition
            else:
                best_loss = acc_loss
                best_partition = partition
    
    return best_partition, best_loss, time.perf_counter()-t0

In [15]:
def validate(priors, thresholds, tt, c):
    p = np.asarray(priors, dtype=float)
    t = np.asarray(thresholds, dtype=float)
    if p.shape != t.shape or p.ndim != 1:
        raise ValueError("priors and thresholds must be 1-D of equal length")
    if np.any(p < 0):
        raise ValueError("priors must be nonnegative")
    if not np.isclose(p.sum(), 1.0):
        raise ValueError(f"priors must sum 1, got {p.sum()}")
    if c <= 0:
        raise ValueError("c must be positive")
    
    return p, t, float(tt), float(c)

def accept_matrix(thresholds):
    t = np.asarray(thresholds, dtype=float)
    return (t[None, :] >= t[:, None]).astype(float)

def build_grid(m, lo=0.0, hi=1.0):
    X = np.linspace(lo, hi, m)
    D = np.full(m, 1.0/m)
    return X, D

def build_constants(priors, thresholds, tt, c, m, eps=1e-6):
    priors, thresholds, tt, c = validate(priors, thresholds, tt, c)
    n = len(thresholds)
    X, D = build_grid(m)

    A = np.empty((m, n+1))
    A[:, 0] = X
    A[:, 1:] = thresholds[None, :]

    cost = c * np.abs(A - X[:, None])
    Hx = (A[:, None, :] >= thresholds[None, :, None]).astype(float)
    U = priors[None, :, None] * (Hx - (1.0 + eps) * cost[:, None, :])
    f = (X >= tt).astype(float)
    L = np.where(f[:, None, None] == 0.0, Hx, 1.0 - Hx)
    M = 1.0 + (1.0 + eps) * cost.max(axis=1)

    valid = A >= X[:, None] - 1e-12
    for xi in range(m):
        seen = set()
        for a in range(n+1):
            key = round(float(A[xi, a]), 12)
            if key in seen:
                valid[xi, a] = False
            else:
                seen.add(key)
    return dict(p=priors, t=thresholds, tt=tt, c=c, n=n, m=m, X=X, D=D, A=A, cost=cost, Hx=Hx, U=U, L=L, M=M, eps=eps, valid=valid)

In [16]:
def get_best_response(K, partition):
    n = K["n"]
    p = np.zeros(n)
    p[partition] = 1
    score = (K["U"] * p[None, :, None]).sum(axis=1)
    k = score.argmax(axis=1)
    X_p = K["A"][np.arange(len(k)), k]
    return X_p

In [17]:
def build_variables(K):
    n, m = K["n"], K["m"]

    xv = cp.Variable((n, n), boolean=True, name="x")
    yv = cp.Variable((m*n, n+1), boolean=True, name="y")

    cons = [
        cp.sum(xv, axis=1) == 1,
        cp.sum(yv, axis=1) == 1,
    ]
    return xv, yv, cons

def row(x_idx, j, n):
    return x_idx * n + j

def assignment_to_partition(assign):
    bins = {}
    for i, j in enumerate(assign):
        bins.setdefault(j, []).append(i)
    return frozenset(frozenset(v) for v in bins.values())

def bell(n):
    r = [1]
    for _ in range(n):
        new = [r[-1]]
        for v in r:
            new.append(new[-1] + v)
        r = new
    return r[0]

In [18]:
def add_best_response(K, xv, yv):
    n, m, U, M, valid = K["n"], K["m"], K["U"], K["M"], K["valid"]
    ones = np.ones((1, n+1))
    cons = []

    for x_idx in range(m):
        util = xv.T @ U[x_idx]
        y_x = yv[row(x_idx, 0, n): row(x_idx, 0, n)+n, :]

        for a in range(n+1):
            if not valid[x_idx, a]:
                cons.append(y_x[:, a] == 0)
                continue
            u_a = cp.reshape(util[:, a], (n,1), order="C")
            y_a = cp.reshape(y_x[:, a], (n,1), order="C")
            cons.append((u_a + M[x_idx] * (1 - y_a)) @ ones >= util)
    return cons

def chosen_landing(K, yv_value):
    n, m = K["n"], K["m"]
    Y = np.asarray(yv_value).reshape(m, n, n+1)
    a = Y.argmax(axis=2)
    return K["A"][np.arange(m)[:, None], a]

def brute_force_landing(K, assign):
    n, m = K["n"], K["m"]
    out = np.zeros((m, n))
    for j in range(n):
        B = (assign == j).astype(float)
        if B.sum() == 0:
            out[:, j] = np.nan
            continue
        score = (K["U"] + B[None, :, None]).sum(axis=1)
        score = np.where(K["valid"], score, -np.inf)
        out[:, j] = K["A"][np.arange(m), score.argmax(axis=1)]
    return out

In [19]:
def add_objective(K, xv, yv):
    n, m , L, D, p = K["n"], K["m"], K["L"], K["D"], K["p"]

    v = cp.Variable((m * n, n), nonneg=True, name="v")
    cons = []
    for x_idx in range(m):
        sl = slice(x_idx * n, (x_idx + 1) * n)
        g = yv[sl, :] @ L[x_idx].T
        cons.append(v[sl, :] >= g + xv.T - 1)

    W = np.repeat(D, n)[:, None] * p[None, :]
    return cp.sum(cp.multiply(W, v)), cons

def evaluate_partition(K, assign):
    n, m, U, L, D, p, valid = K["n"], K["m"], K["U"], K["L"], K["D"], K["p"], K["valid"]

    total = 0.0
    for j in range(n):
        members = np.flatnonzero(assign == j)
        if members.size == 0:
            continue
        B = np.zeros(n)
        B[members] = 1.0
        score = (U * B[None, :, None]).sum(axis=1)
        a_star = np.where(valid, score, -np.inf).argmax(axis=1)
        err = L[np.arange(m)[:, None], members[None, :], a_star[:, None]]
        total += float(D @ (err @ p[members]))
    return total

def add_symmetry(K, xv):
    n = K["n"]
    cons = [xv[i, j] == 0 for i in range(n) for j in range(i+1, n)]
    for i in range(1, n):
        for j in range(1, i+1):
            cons.append(xv[i,j] <= cp.sum(xv[:i, j-1]))
    return cons

def pin_empty_bins(K, xv, yv):
    n = K["n"]
    occ = cp.sum(xv, axis=0)
    return [yv[x_idx * n:(x_idx + 1) * n, 0] >= 1 - occ for x_idx in range(K["m"])]

def solve(priors, thresholds, tt, c, m=51, eps=1e-6, prune=True, pin_empty=True, feas_tol=1e-9, check_tol=1e-9, time_limit=None):
    K = build_constants(priors, thresholds, tt, c, m, eps)
    xv, yv, cons = build_variables(K)
    cons = cons + add_best_response(K, xv, yv)
    if prune:
        cons = cons + add_symmetry(K, xv)
    if pin_empty:
        cons = cons + pin_empty_bins(K, xv, yv)
    obj, obj_cons = add_objective(K, xv, yv)

    prob = cp.Problem(cp.Minimize(obj), cons + obj_cons)
    kw = dict(primal_feasibility_tolerance=feas_tol, mip_feasibility_tolerance=feas_tol)
    if time_limit is not None:
        kw["time_limit"] = time_limit
    
    t0 = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        prob.solve(solver=cp.HIGHS, **kw)
    elapsed = time.perf_counter() - t0

    assign = np.asarray(xv.value).argmax(axis=1)
    loss = evaluate_partition(K, assign)
    return dict(
        status=prob.status,
        partition=sorted(sorted(b) for b in assignment_to_partition(assign)),
        loss=loss,
        solver_objective=float(prob.value),
        certified=bool(loss - float(prob.value) < check_tol),
        seconds=elapsed,
        K=K
    )

def brute_force(K):
    t0 = time.perf_counter()
    n = K["n"]
    best = (np.inf, None)
    for assign in itertools.product(range(n), repeat=n):
        val = evaluate_partition(K, np.array(assign))
        if val < best[0] - 1e-12:
            best = (val, np.array(assign))
    return best, time.perf_counter() - t0

In [20]:
m = 11
n = 15
thresholds = np.linspace(0.0, 1.0, n)
priors = np.full(n, 1/n)
c = 1.
tt = 0.1

print(f"m: {m} | n: {n}")
for prune in [True]:
	r = solve(priors, thresholds, tt, c, m, prune=prune)
	print(f"[Prune: {str(prune):10}]  Time: {r['seconds']:7.2f}s  | Loss: {r['loss']:.6f}  | Partition: {r['partition']}  | Solver Obj: {r['solver_objective']:.6f}")

# opt_partition, opt_loss, opt_time = find_partitions_optimal(r["K"]["X"], thresholds, priors, tt, c)
# print(f"[Mode : {'BF':10}]  Time: {opt_time:7.2f}s  | Loss: {opt_loss:.6f}  | Partition: {opt_partition}")

# (bf_loss, bf_assign), bf_sec = brute_force(r["K"])
# bf_partition = sorted(sorted(b) for b in assignment_to_partition(bf_assign))
# print(f"[Mode: {'BF':10}]  Time: {bf_sec:7.2f}s  | Loss: {bf_loss:.6f}  | Partition: {bf_partition}")

m: 11 | n: 15
[Prune: True      ]  Time:    5.80s  | Loss: 0.012121  | Partition: [[0, 1], [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]]  | Solver Obj: 0.012121


In [21]:
def generate_prior_grid(n_components, n_balls=50):
    step = 1.0 / n_balls
    grids = []
    for combo in itertools.combinations_with_replacement(range(n_components), n_balls - n_components):
        counts = np.bincount(combo, minlength=n_components) + 1
        grids.append((counts * step).round(4))
    return grids

In [ ]:
M = [11]
N = [8, 10]
# prune = [False, True]
prune = [True]
tt = 0.1
c = 1.0
for m in M:
    for n in N:
        results = {"i": [], "alg": [], "time": [], "loss": [], "partition": []}
        thresholds = np.linspace(0.0, 1.0, n)
        prior_grid = generate_prior_grid(n, 20)
        # if n >= 8:
        #     prune_n = prune[1:]
        # else:
        #     prune_n = prune
        for i, priors in tqdm.tqdm(enumerate(prior_grid), desc=f"[m={m}] [n={n}]", total=len(prior_grid)):
            # for p in prune_n:
            milp = solve(priors, thresholds, tt, c, m, prune=True, time_limit=None)
            results["i"].append(i)
            # results["alg"].append("base" if not p else "pruned")
            results["alg"].append("pruned")
            results["time"].append(milp["seconds"])
            results["loss"].append(milp["loss"])
            results["partition"].append(milp["partition"])
            
            opt_partition, opt_loss, opt_time = find_partitions_optimal(milp["K"]["X"], thresholds, priors, tt, c)
            results["i"].append(i)
            results["alg"].append("brute force")
            results["time"].append(opt_time)
            results["loss"].append(opt_loss)
            results["partition"].append(opt_partition)
        
        df = pd.DataFrame(results)
        df.to_pickle(f"grid_search_solver_m{m}_n{n}.pkl")

[m=11] [n=10]:   0%|          | 70/92378 [19:20<400:10:29, 15.61s/it]